In [3]:
from pathlib import Path
from datasets import load_dataset

# --- Config ---
N = 200  # total samples to label (increase to continue further)
LEGIBLE_DIR = Path("../data/partitions/clear")
ILLEGIBLE_DIR = Path("../data/partitions/unclear")
PROGRESS_FILE = Path("../data/partitions/progress.txt")

LEGIBLE_DIR.mkdir(parents=True, exist_ok=True)
ILLEGIBLE_DIR.mkdir(parents=True, exist_ok=True)

ds = load_dataset("deepcopy/MathWriting-human")
ds_val = ds["val"].select(range(N))

start_idx = 0
if PROGRESS_FILE.exists():
    start_idx = int(PROGRESS_FILE.read_text().strip())
    print(f"Resuming from sample {start_idx}")

print(f"Loaded {N} samples. Starting at index {start_idx}.")

Resuming from sample 200
Loaded 200 samples. Starting at index 200.


In [ ]:
from IPython.display import display, clear_output

for idx in range(start_idx, N):
    clear_output(wait=True)
    sample = ds_val[idx]
    print(f"=== Sample {idx + 1} / {N} ===")
    print(f"GT: {sample['latex'][:120]}")
    display(sample["image"])
    
    while True:
        choice = input("1 = Clear, 2 = Unclear, u = Undo, q = Quit: ").strip().lower()
        if choice in ("1", "2", "u", "q"):
            break
        print("Invalid input. Try again.")
    
    if choice == "q":
        PROGRESS_FILE.write_text(str(idx))
        print(f"Saved progress at sample {idx}. Run cell again to resume.")
        break
    elif choice == "u":
        if idx > start_idx:
            # Re-do previous sample by adjusting — but since we're in a for loop,
            # we can't go back. Save progress and tell user to re-run.
            PROGRESS_FILE.write_text(str(max(0, idx - 1)))
            print(f"Undo: re-run this cell to redo sample {idx - 1}.")
            break
    else:
        dest = LEGIBLE_DIR if choice == "1" else ILLEGIBLE_DIR
        other = ILLEGIBLE_DIR if choice == "1" else LEGIBLE_DIR
        sample["image"].save(dest / f"{idx:05d}.png")
        other_path = other / f"{idx:05d}.png"
        if other_path.exists():
            other_path.unlink()
        PROGRESS_FILE.write_text(str(idx + 1))
else:
    clear_output(wait=True)
    print(f"Done! All {N} samples labeled.")
    legible_count = len(list(LEGIBLE_DIR.glob("*.png")))
    illegible_count = len(list(ILLEGIBLE_DIR.glob("*.png")))
    print(f"Legible: {legible_count}, Illegible: {illegible_count}")

Done! All 200 samples labeled.
Legible: 188, Illegible: 12
